# 05 — GATE RUN: ImageNet-1k, banyak skenario sekaligus

**Ini RUN GERBANG, bukan run paper.** Ia bisa gagal dan menghentikan pengembangan metode.
Kriterianya ditetapkan di `reports/prereg_imagenet_gate.md`, **ditulis sebelum data diunduh**.
Baca dokumen itu dulu — notebook ini tidak boleh dipakai untuk memilih kriteria.

## Mengapa ImageNet, dan mengapa ini penyimpangan

Urutan pre-registered menyatakan dataset berikutnya hanya dijalankan jika Phase 1 lulus di
Pl@ntNet. **Gate B/C tidak lulus.** Jadi ini penyimpangan, dicatat sebagai keputusan eksplisit.

Alasannya daya uji, dan itu terukur tanpa melihat hasil Pl@ntNet:

| dump skor | sampel x kelas | per kelas | kelas dgn delta_y (n_cal=25) |
|---|---|---|---|
| LTC plantnet cal | 21.783 x 1.081 | median **3** | **152** -> 38 per stratum |
| CCC imagenet | 115.301 x 1.000 | **115** | **1.000**, tanpa stratifikasi |

Uji permutasi tingkat-kelas, terkalibrasi: pada 38 kelas sinyal lemah memberi p=0,066 (tidak
terdeteksi); pada 1.000 kelas sinyal lemah yang sama memberi p=0,0066.

ImageNet juga **berimbang**, jadi confound kualitas-deskriptor <-> prevalensi yang memaksa
Amandemen 5 tidak ada di sini — stratifikasi tidak diperlukan, bukan dihindari.

## Yang dibundel dalam satu run

gate A - gate B (bootstrap tingkat-KELAS) - gate C (permutasi tingkat-KELAS + Holm) -
multi-alpha {0,01; 0,05; 0,10} - dua n_cal - Sec 6.4 - reproduksi Clustered CP - perbandingan
berdampingan dengan Pl@ntNet.

## Yang HARUS diingat saat membaca hasilnya

phi(y) di sini adalah geometri **ruang OUTPUT**, dihitung dari matriks skor saja — bukan phi(y)
embedding yang dipakai di Pl@ntNet. Hasil di sini **tidak otomatis berpindah** ke deskriptor
embedding. Yang ia jawab: apakah geometri tingkat-kelas memprediksi delta_y begitu jumlah
kelasnya memadai.


## 1. Config — `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
SOURCE = 'ltc'                 # 'ltc' (otomatis) | 'ccc' (satu langkah manual)
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
DATASET  = 'imagenet'          # dipakai hanya kalau SOURCE == 'ccc'
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump
# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Unduh dump skor — TANPA citra, TANPA GPU

LTC memakai ID gdown yang sudah kita punya dari notebook 00, jadi tidak ada langkah manual.
Tidak ada forward pass di seluruh notebook ini, sehingga gate checkpoint tidak berlaku:
metriknya dihitung dari array rilis, bukan dari forward pass kami.


In [ ]:
import glob, numpy as np
SCORE_DIR = f'{DRIVE_ROOT}/scores_{SOURCE}'
os.makedirs(SCORE_DIR, exist_ok=True)

if SOURCE == 'ltc':
    import gdown
    for ds in LTC_DATASETS:
        tgt = f'{SCORE_DIR}/{ds}'
        if glob.glob(f'{tgt}/**/*.npy', recursive=True):
            print(f'{ds}: sudah ada, dilewati')
            continue
        os.makedirs(tgt, exist_ok=True)
        print(f'{ds}: mengunduh...')
        ok = False
        try:
            gdown.download_folder(id=GID_SCORES_LTC[ds], output=tgt, quiet=True,
                                  use_cookies=False)
            ok = True
        except Exception as e:
            print(f'  download_folder gagal ({type(e).__name__}); coba sebagai berkas tunggal')
        if not ok:
            try:
                gdown.download(id=GID_SCORES_LTC[ds], output=f'{tgt}/scores.zip', quiet=True)
                subprocess.run(['unzip','-q','-o',f'{tgt}/scores.zip','-d',tgt], check=False)
            except Exception as e2:
                print(f'  GAGAL juga: {type(e2).__name__}: {e2}')
else:
    print('SOURCE=ccc -> pakai sel manual di bawah untuk membaca ID dari download_data.sh')

found = sorted(glob.glob(f'{SCORE_DIR}/**/*.npy', recursive=True))
print(f'total .npy di {SCORE_DIR}: {len(found)}')
for f in found[:40]:
    print('  ', os.path.relpath(f, SCORE_DIR))


### 3b. Fallback CCC — hanya jika tidak ada dump LTC yang memenuhi premis


In [ ]:
DO_CCC = False   # set True hanya kalau sel 4 melaporkan PREMIS TIDAK TERPENUHI
if DO_CCC:
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/tiffanyding/class-conditional-conformal.git',
                    '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('--- jalankan perintah gdown di atas dengan tujuan', SCORE_DIR, '---')
else:
    print('dilewati')


## 4. SURVEI semua dump — mana yang punya daya uji?

Ini menjawab pertanyaan dataset secara **empiris**, bukan dengan inferensi. Yang menentukan
daya uji gate C bukan jumlah kelas, melainkan **jumlah kelas yang punya cukup sampel
KALIBRASI**. Setiap dump disurvei, lalu yang terbaik dipakai. Kalau tidak ada yang memenuhi
premis, itu TEMUAN dan run berhenti di sini.


In [ ]:
import numpy as np
from pcc.data.load import softmax_from_logits

# n_cal primer harus tercapai DI DALAM porsi CAL, jadi totalnya harus lebih besar
NEED_TOTAL = int(np.ceil(N_CAL_PRIMARY / FRAC_CAL))

def survey_pair(p2, p1):
    S = np.load(p2, mmap_mode='r'); y = np.load(p1).astype(int)
    if len(y) != len(S) or y.min() < 0 or y.max() >= S.shape[1]:
        return None
    c = np.bincount(y, minlength=S.shape[1])
    return dict(scores=p2, labels=p1, n=len(y), K=int(S.shape[1]),
                med=float(np.median(c)), empty=int((c == 0).sum()),
                n_feasible=int((c >= NEED_TOTAL).sum()))

cands = []
for f2 in found:
    a = np.load(f2, mmap_mode='r')
    if a.ndim != 2:
        continue
    for f1 in found:
        b = np.load(f1, mmap_mode='r')
        if b.ndim == 1 and len(b) == len(a) and os.path.dirname(f1) == os.path.dirname(f2):
            r = survey_pair(f2, f1)
            if r:
                cands.append(r)
            break

assert cands, 'tidak ada pasangan (skor 2-D, label 1-D) yang cocok — cek isi SCORE_DIR'
hdr = 'dump'.ljust(46) + 'n'.rjust(9) + 'K'.rjust(7) + 'med/kls'.rjust(9) + 'layak'.rjust(8)
print(hdr)
for r in sorted(cands, key=lambda r: -r['n_feasible']):
    nm = os.path.relpath(r['scores'], SCORE_DIR)[:44]
    print(nm.ljust(46) + str(r['n']).rjust(9) + str(r['K']).rjust(7)
          + f"{r['med']:9.0f}" + str(r['n_feasible']).rjust(8))

best = max(cands, key=lambda r: r['n_feasible'])
PREMISE_OK = best['n_feasible'] >= 500
print()
print('terbaik:', os.path.relpath(best['scores'], SCORE_DIR))
print(f"  {best['n_feasible']} kelas punya >= {NEED_TOTAL} sampel "
      f"(agar {N_CAL_PRIMARY} tercapai di porsi CAL {FRAC_CAL:.0%})")
print('PREMIS PRE-DEKLARASI (>=500 kelas layak):',
      'TERPENUHI' if PREMISE_OK else 'TIDAK TERPENUHI')
if not PREMISE_OK:
    print()
    print('  TEMUAN: tidak ada dump LTC yang memberi daya uji yang dibutuhkan.')
    print('  Artinya batas 38-kelas/stratum di Pl@ntNet BUKAN kekhususan Pl@ntNet —')
    print('  ia berlaku untuk keluarga dataset ekor-panjang yang dipakai prior art.')
    print('  JANGAN tafsirkan gate B/C dari run ini. Set DO_CCC=True di sel 3b,')
    print('  ambil dump ImageNet CCC, set SOURCE=ccc, lalu ulangi.')

# float32: dump besar x salinan thr_lac bisa mencapai beberapa GB, dan sesi ini pernah
# crash karena alokasi besar. float64 hanya perlu saat mereproduksi softmax rilis
# bit-per-bit, yang tidak dilakukan di sini.
S_all = np.load(best['scores']).astype(np.float32)
y_all = np.load(best['labels']).astype(int)
K = S_all.shape[1]
if N_CLASSES_EXPECTED is not None:
    assert K == N_CLASSES_EXPECTED, f'kelas {K}, diharapkan {N_CLASSES_EXPECTED}'
print()
print(f'dipakai: {S_all.shape} {S_all.dtype} ({S_all.nbytes/1e6:.0f} MB)')

row = S_all[:200].sum(axis=1)
is_prob = bool(np.allclose(row, 1.0, atol=1e-3))
print(f'baris berjumlah 1 (sudah softmax): {is_prob}')
if not is_prob:
    print('  -> diperlakukan sebagai logits, dikonversi')
    S_all = softmax_from_logits(S_all)
cnt = np.bincount(y_all, minlength=K)
acc = float((S_all.argmax(axis=1) == y_all).mean())
print(f'akurasi top-1: {acc:.4f} | kelas kosong: {(cnt==0).sum()}')


## 5. Tiga split terpisah — leak guard ditegakkan

DESC 40% (phi saja) / CAL 30% (delta_y) / EVAL 30% (semua metrik). Stratified per kelas.


In [ ]:
rng = np.random.default_rng(SEED)
role = np.empty(len(y_all), dtype='<U4')
for c in range(K):
    idx = np.where(y_all == c)[0]
    rng.shuffle(idx)
    n = len(idx); n_d = int(round(FRAC_DESC*n)); n_c = int(round(FRAC_CAL*n))
    role[idx[:n_d]] = 'desc'
    role[idx[n_d:n_d+n_c]] = 'cal'
    role[idx[n_d+n_c:]] = 'eval'

i_desc = np.where(role == 'desc')[0]
i_cal  = np.where(role == 'cal')[0]
i_eval = np.where(role == 'eval')[0]
assert len(set(i_desc) & set(i_cal)) == 0
assert len(set(i_cal) & set(i_eval)) == 0
assert len(set(i_desc) & set(i_eval)) == 0
assert len(i_desc) + len(i_cal) + len(i_eval) == len(y_all)
print(f'DESC {len(i_desc)}  CAL {len(i_cal)}  EVAL {len(i_eval)}  (terpisah, terverifikasi)')

cnt_cal = np.bincount(y_all[i_cal], minlength=K)
cnt_desc = np.bincount(y_all[i_desc], minlength=K)
print(f'CAL sampel/kelas: median {np.median(cnt_cal):.0f} min {cnt_cal.min()}')
print(f'kelas dengan >= {N_CAL_PRIMARY} sampel CAL: {(cnt_cal >= N_CAL_PRIMARY).sum()}/{K}')


## 6. phi(y) dari ruang OUTPUT — hanya dari split DESC


In [ ]:
from pcc.descriptors.output_space import build_output_descriptors, QUOTA_DETERMINED

Phi, names = build_output_descriptors(S_all[i_desc], y_all[i_desc], K,
                                     log_prevalence_from=cnt_desc)
print(f'Phi {Phi.shape} | baris finite: {int(np.isfinite(Phi).all(axis=1).sum())}/{K}')
print('fitur:', names)
print('QUOTA_DETERMINED (tidak boleh dikreditkan stabilitas):', QUOTA_DETERMINED)


## 7. Screen stabilitas — set PRIMER = fitur dengan stabilitas >= 0,90

Dihitung dengan membelah split DESC jadi dua bagian terpisah dan mengorelasikan phi lintas
kelas. Prosedurnya sama seperti Pl@ntNet, jadi ambangnya bisa dibandingkan.


In [ ]:
half = {}
r2 = np.random.default_rng(SEED + 7)
mask = np.zeros(len(i_desc), bool)
for c in range(K):
    loc = np.where(y_all[i_desc] == c)[0]
    r2.shuffle(loc)
    mask[loc[:len(loc)//2]] = True
PhiA, _ = build_output_descriptors(S_all[i_desc][mask],  y_all[i_desc][mask],  K,
                                  log_prevalence_from=cnt_desc)
PhiB, _ = build_output_descriptors(S_all[i_desc][~mask], y_all[i_desc][~mask], K,
                                  log_prevalence_from=cnt_desc)
stab = {}
for j, nm in enumerate(names):
    a, b = PhiA[:, j], PhiB[:, j]
    ok = np.isfinite(a) & np.isfinite(b)
    stab[nm] = (float(np.corrcoef(a[ok], b[ok])[0, 1])
                if ok.sum() > 3 and np.std(a[ok]) > 0 and np.std(b[ok]) > 0 else np.nan)
for nm in sorted(stab, key=lambda n: -(stab[n] if np.isfinite(stab[n]) else -9)):
    tag = ' [QUOTA]' if nm in QUOTA_DETERMINED else ''
    print(f'  {nm:18s} {stab[nm]:+.3f}{tag}')

stable_names = [n for n in names
                if n not in QUOTA_DETERMINED
                and np.isfinite(stab[n]) and stab[n] >= STABLE_THRESHOLD]
print()
print(f'lolos screen ({len(stable_names)}):', stable_names)

# prof_knn_1 DICADANGKAN sebagai baseline jarak pre-registered dan DIKELUARKAN dari
# model penuh. Kalau ia ikut di dalam model, gate C hanya menguji apakah fitur sisanya
# menambah sesuatu di atas prof_knn_1 — submodel bersarang, bukan perbandingan
# terhadap baseline prior-art gaya Fargion yang dimaksud Sec 6.5C.
DISTANCE_BASELINE = 'prof_knn_1'
stable_names = [n for n in stable_names if n != DISTANCE_BASELINE]
full_names = [n for n in names if n != DISTANCE_BASELINE]
print(f'baseline jarak dicadangkan: {DISTANCE_BASELINE} (di luar model penuh)')
print(f'set PRIMER stable ({len(stable_names)}):', stable_names)
print('Sec 3.3 terpenuhi:', bool(stable_names))
assert stable_names, 'tidak ada fitur lolos screen — laporkan, jangan turunkan ambang'

lp = Phi[:, names.index('log_prevalence')]
lp = lp[np.isfinite(lp)]
vac = np.std(lp) < 1e-8
print(f'log_prevalence: sd={np.std(lp):.6f} rentang={np.ptp(lp):.6f}')
print('  ablasi prevalensi:', 'HAMPA (dikeluarkan dari verdict)' if vac else 'bisa diuji')
FEATURE_SETS = {'stable': stable_names, 'full': full_names}


## 8. Gate A — reliabilitas delta_y, dan plafonnya


In [ ]:
from pcc.scores.base import thr_lac
from pcc.targets.delta import split_half_reliability, delta_y_matched_n
from pcc.eval.stats import mean_ci

S_cal = thr_lac(S_all[i_cal]); y_cal = y_all[i_cal]
s_true_cal = S_cal[np.arange(len(y_cal)), y_cal]
S_ev = thr_lac(S_all[i_eval]); y_ev = y_all[i_eval]

relA = split_half_reliability(s_true_cal, y_cal, K, ALPHA_PRIMARY,
                              n_splits=N_SPLITS_A, seed=SEED)
r_delta = float(relA['reliability_mean'])
ciA = mean_ci(relA['reliability_splits'])
print(f"gate A r_delta = {r_delta:.3f}  CI [{ciA['ci_low']:.3f}, {ciA['ci_high']:.3f}]")
print(f"  kelas eligible: {relA['n_classes_eligible']}  "
      f"split bernilai: {relA['n_splits_with_a_value']}/{N_SPLITS_A}")
gate_A_pass = bool(np.isfinite(ciA['ci_low']) and ciA['ci_low'] >= 0.30)
print('gate A:', 'LULUS' if gate_A_pass else 'GAGAL')

r_phi = float(np.nanmean([stab[n] for n in stable_names]))
print(f'r_phi (rata-rata stabilitas set stable) = {r_phi:.3f}')
print(f'plafon gabungan r_delta*r_phi = {r_delta*r_phi:.3f}')


## 9. delta_y pada n_cal tercocokkan + null prevalensi

ImageNet berimbang, jadi `log n_y` nyaris konstan dan null prevalensi kemungkinan **tidak
terdefinisi** — itu diharapkan dan dilaporkan, bukan error.


In [ ]:
from pcc.targets.delta import prevalence_null

deltas = {}
for tag, ncal in (('primary', N_CAL_PRIMARY), ('secondary', N_CAL_SECONDARY)):
    d, kept = delta_y_matched_n(s_true_cal, y_cal, K, ALPHA_PRIMARY, n_cal=ncal, seed=SEED)
    deltas[tag] = (d, kept, ncal)
    nz = int(np.isfinite(d).sum())
    print(f'[{tag}] n_cal={ncal}: delta_y terdefinisi untuk {nz}/{K} kelas '
          f'(sd {np.nanstd(d):.4f})')

pn = prevalence_null(s_true_cal, y_cal, K, ALPHA_PRIMARY, n_cal=N_CAL_PRIMARY,
                     n_reps=30, seed=SEED)
print('null prevalensi:', pn.get('undefined_reason') or
      f"mean {pn['null_mean']:+.3f} sd {pn['null_sd']:.3f}")


## 10. UJI PRIMER — gate B (bootstrap KELAS) dan gate C (permutasi KELAS + Holm)

Kriteria yang lama, *CI selisih mengecualikan 0* atas sebaran antar-split, **bukan uji yang
valid**: nol bukan nilai null-nya (terukur: 38 kelas tanpa sinyal memberi selisih +0,109
sementara null berpusat di -0,102), dan split adalah pemakaian ulang kelas yang sama, bukan
observasi independen. Unit yang bisa ditukar adalah **kelas**.


In [ ]:
from pcc.eval.predictability import (predictability, predictability_class_bootstrap,
                                     class_permutation_p)
from pcc.eval.stats import holm_bonferroni

d_prim = deltas['primary'][0]
feats = FEATURE_SETS['stable']

boot = predictability_class_bootstrap(Phi, d_prim, names, feature_subset=feats,
                                     n_boot=N_BOOT_CLASS, n_splits=10, seed=SEED,
                                     reliability=r_delta)
print('GATE B — bootstrap tingkat-KELAS (PRIMER)')
print(f"  R2 {boot['mean']:+.4f}  CI [{boot['ci_low']:+.4f}, {boot['ci_high']:+.4f}]"
      f"  n_kelas={boot['n_classes']} n_boot={boot['n_boot']}")
if 'normalized_mean' in boot:
    print(f"  ter-normalisasi {boot['normalized_mean']:+.4f} "
          f"CI [{boot['normalized_ci_low']:+.4f}, {boot['normalized_ci_high']:+.4f}]")
gate_B_pass = bool(boot.get('gate_B_pass_class_unit'))
print('  gate B:', 'LULUS' if gate_B_pass else 'GAGAL')

spl = predictability(Phi, d_prim, names, feature_subset=feats, reliability=r_delta,
                     n_splits=N_SPLITS_BC, seed=SEED)
sf = spl['r2_by_predictor']['full']
print(f"  [sekunder, CI antar-split] R2 {sf['mean']:+.4f} "
      f"[{sf['ci_low']:+.4f}, {sf['ci_high']:+.4f}]")
print(f"  baseline jarak: {spl['distance_col_used']} "
      f"(bersarang di full: {spl['distance_baseline_is_nested_in_full']})")


In [ ]:
# Keluarga uji PRIMER. Ablasi prevalensi hanya masuk kalau ia BUKAN hampa —
# memasukkan uji yang dijamin menang akan MENGENCERKAN koreksi Holm dan membuat
# gate C lebih mudah, bukan lebih ketat.
ABLATIONS_PRIMARY = ['distance_only']
if not spl['prevalence_ablation_degenerate']:
    ABLATIONS_PRIMARY.append('log_prevalence_only')
else:
    print('CATATAN: ablasi prevalensi HAMPA (dataset berimbang) — dikeluarkan dari')
    print('  keluarga uji primer. Gate C di sini menguji HANYA lawan baseline jarak:')
    print('  lengan prior-art yang lebih penting, tetapi uji yang LEBIH SEMPIT')
    print('  daripada di Pl@ntNet, dan harus dilaporkan sebagai lebih sempit.')
print(f'GATE C — null permutasi tingkat-KELAS, n_perm={N_PERM_CLASS} (PRIMER)')
perm_res, pvals, labels = {}, [], []
for abl in ABLATIONS_PRIMARY:
    if abl not in spl['gate_C_detail']:
        print(f'  {abl}: tidak tersedia di set fitur ini — dilewati')
        continue
    r = class_permutation_p(Phi, d_prim, names, feature_subset=feats, ablation=abl,
                            n_perm=N_PERM_CLASS, n_splits=20, seed=SEED)
    perm_res[abl] = r
    pvals.append(r['p_value']); labels.append(abl)
    print(f"  {abl:24s} obs {r['observed']:+.4f} null {r['null_mean']:+.4f}"
          f" (sd {r['null_sd']:.4f})  p={r['p_value']:.4f}")

holm = holm_bonferroni(pvals, alpha=0.05) if pvals else None
gate_C_pass = False
if holm:
    print()
    print('  Holm-Bonferroni lintas keluarga uji (Sec 8.6):')
    for k, h in zip(labels, holm):
        pv = h['p_value']; th = h['threshold']; rj = h['reject']
        print(f'    {k:24s} p={pv:.4f} ambang={th:.4f} tolak_H0={rj}')
    gate_C_pass = all(h['reject'] for h in holm)
print('  gate C:', 'LULUS' if gate_C_pass else 'GAGAL')


## 11. SEKUNDER — multi-alpha (Sec 8.8), yang Pl@ntNet tidak pernah bisa

Dengan ~115 sampel/kelas, alpha=0,01 butuh >=99 dan **terpenuhi**. Di Pl@ntNet alpha=0,01
hanya layak untuk 57 dari 1.081 kelas.


In [ ]:
multi = {}
for a in (ALPHA_PRIMARY,) + tuple(ALPHAS_SECONDARY):
    need = int(np.ceil(1/a)) - 1
    feasible = int((cnt_cal >= need).sum())
    st = thr_lac(S_all[i_cal])[np.arange(len(y_cal)), y_cal]
    d_a, _ = delta_y_matched_n(st, y_cal, K, a, n_cal=N_CAL_PRIMARY, seed=SEED)
    nz = int(np.isfinite(d_a).sum())
    if nz < 50:
        multi[a] = {'skipped': True, 'n_classes': nz}
        print(f'alpha={a}: hanya {nz} kelas — dilewati')
        continue
    b = predictability_class_bootstrap(Phi, d_a, names, feature_subset=feats,
                                       n_boot=200, n_splits=8, seed=SEED,
                                       reliability=r_delta)
    multi[a] = {'n_classes': nz, 'feasible_classes': feasible,
                'r2': b['mean'], 'ci': [b['ci_low'], b['ci_high']],
                'gate_B': bool(b.get('gate_B_pass_class_unit'))}
    tag = ' <- PRIMER' if a == ALPHA_PRIMARY else ''
    print(f"alpha={a}: kelas layak {feasible}/{K}, delta_y {nz}, "
          f"R2 {b['mean']:+.4f} [{b['ci_low']:+.4f},{b['ci_high']:+.4f}] "
          f"gate_B={multi[a]['gate_B']}{tag}")


## 12. SEKUNDER — set `full`, n_cal=50, dan Sec 6.4 (Amandemen 8)


In [ ]:
sec = {}
for fset in ('stable', 'full'):
    b = predictability_class_bootstrap(Phi, d_prim, names,
                                       feature_subset=FEATURE_SETS[fset],
                                       n_boot=200, n_splits=8, seed=SEED,
                                       reliability=r_delta)
    sec[f'{fset}|n_cal{N_CAL_PRIMARY}'] = b
    print(f"[{fset}] p={len(FEATURE_SETS[fset])} R2 {b['mean']:+.4f} "
          f"[{b['ci_low']:+.4f},{b['ci_high']:+.4f}] gate_B={b.get('gate_B_pass_class_unit')}")

d_sec = deltas['secondary'][0]
b2 = predictability_class_bootstrap(Phi, d_sec, names, feature_subset=feats,
                                    n_boot=200, n_splits=8, seed=SEED,
                                    reliability=r_delta)
sec[f'stable|n_cal{N_CAL_SECONDARY}'] = b2
print(f"[stable, n_cal={N_CAL_SECONDARY}] R2 {b2['mean']:+.4f} "
      f"[{b2['ci_low']:+.4f},{b2['ci_high']:+.4f}] gate_B={b2.get('gate_B_pass_class_unit')}")


In [ ]:
from pcc.eval.predictability import ridge_fit, ridge_predict
from pcc.eval.setsize import setsize_translation_shrunk
from pcc.eval.decomposition import group_quantile

cols = [names.index(f) for f in feats]
usable = np.where(np.isfinite(d_prim) & np.isfinite(Phi[:, cols]).all(axis=1))[0]
print(f'kelas usable untuk Sec 6.4: {len(usable)}')
qg = group_quantile(s_true_cal, ALPHA_PRIMARY, 'empirical')
rg = np.random.default_rng(SEED)
acc64 = {'obs': [], 'null': [], 'oracle': [], 'raw': [], 'lam': []}
for rep in range(20):
    perm = rg.permutation(usable)
    fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
    m = ridge_fit(Phi[fit_c][:, cols], d_prim[fit_c], 1.0)
    dh = np.zeros(K)
    dh[held_c] = ridge_predict(m, Phi[held_c][:, cols])
    dh[fit_c]  = ridge_predict(m, Phi[fit_c][:, cols])
    dn = np.array(dh); dn[held_c] = rg.permutation(dh[held_c])
    for tag, dd in (('obs', dh), ('null', dn)):
        try:
            r = setsize_translation_shrunk(S_ev, y_ev, ALPHA_PRIMARY, None, None,
                                           fit_c, held_c, dd, stat='worst', q_global=qg)
        except ValueError:
            continue
        acc64[tag].append(r['delta']['worst'])
        if tag == 'obs':
            acc64['lam'].append(r['lambda_selected_on_train'])
            acc64['oracle'].append(r['controls']['oracle_ceiling'])
            acc64['raw'].append(r['controls']['raw_delta_lambda1'])
o = mean_ci(np.array(acc64['obs'], float)); n0 = mean_ci(np.array(acc64['null'], float))
orc = mean_ci(np.array(acc64['oracle'], float)); raw = mean_ci(np.array(acc64['raw'], float))
sec64 = {'observed': o, 'shuffled_null': n0, 'oracle_ceiling': orc,
         'raw_delta_lambda1': raw,
         'lambda_mean': float(np.mean(acc64['lam'])) if acc64['lam'] else float('nan'),
         'beats_null': bool(o['ci_low'] > n0['ci_high']),
         'pass': bool(o['ci_low'] > 0 and o['ci_low'] > n0['ci_high'])}
print(f"Sec 6.4: observed {o['mean']:+.4f} [{o['ci_low']:+.4f},{o['ci_high']:+.4f}]")
print(f"  null {n0['mean']:+.4f} | ORACLE {orc['mean']:+.4f} | raw lam=1 {raw['mean']:+.4f}")
print(f"  lambda dari TRAIN {sec64['lambda_mean']:.3f} -> "
      f"{'LULUS' if sec64['pass'] else 'TIDAK POSITIF'}")
if orc['mean'] <= 0.02:
    print('  PERINGATAN: tanpa ruang oracle, metriknya tidak bisa positif — jangan dibaca')


## 13. SEKUNDER — reproduksi Clustered CP pada skor yang SAMA

Pemeriksaan kesetiaan setup. Kalau reproduksi kita cocok dengan angka terbit Ding et al.,
sitiran jadi berlandas; kalau tidak, ketidakcocokan itu sendiri yang dilaporkan.


In [ ]:
clustered = None
if RUN_CLUSTERED_CP:
    try:
        import sys
        if not os.path.isdir('/content/ccc'):
            subprocess.run(['git','clone','--depth','1',
                            'https://github.com/tiffanyding/class-conditional-conformal.git',
                            '/content/ccc'], check=True)
        sys.path.insert(0, '/content/ccc')
        from utils.clustering_utils import clustered_conformal
        print('impor clustered_conformal: OK — jalankan pada (S_cal, y_cal) -> (S_ev, y_ev)')
        print('CATATAN: signature-nya milik mereka; sesuaikan pemanggilan sesuai example.ipynb')
        clustered = {'imported': True}
    except Exception as e:
        clustered = {'imported': False, 'error': f'{type(e).__name__}: {e}'}
        print('gagal impor:', clustered['error'])
        print('Bukan pemblokir gerbang — dilaporkan sebagai item terbuka.')
else:
    print('dilewati')


## 14. VERDICT dan laporan


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return clean(o.tolist())
    return o

print('=== UJI PRIMER (prereg_imagenet_gate.md) ===')
print('  gate A :', 'LULUS' if gate_A_pass else 'GAGAL')
print('  gate B :', 'LULUS' if gate_B_pass else 'GAGAL', '(bootstrap tingkat-kelas)')
print('  gate C :', 'LULUS' if gate_C_pass else 'GAGAL', '(permutasi kelas + Holm)')
print('  Sec 6.4:', 'LULUS' if sec64['pass'] else 'TIDAK POSITIF', '(sekunder)')
print()
if gate_B_pass and gate_C_pass:
    verdict = 'LULUS'
    print('KONSEKUENSI (sudah ditetapkan): kegagalan Pl@ntNet TERKONFIRMASI sebagai daya uji.')
    print('Phase 1 dicatat lulus; lanjut ke phi embedding lalu Phase 2.')
elif gate_B_pass:
    verdict = 'GATE C GAGAL'
    print('KONSEKUENSI: delta_y terprediksi tetapi TIDAK melampaui prediktor trivial.')
    print('Sec 6.5C terpicu — pertimbangkan menghentikan klaim kebaruan.')
else:
    verdict = 'GATE B GAGAL'
    print('KONSEKUENSI: delta_y tidak terprediksi bahkan pada jumlah kelas ini.')
    print('Batasan sejati; tulis hasil negatif.')
print()
print('INGAT: phi di sini ruang OUTPUT, bukan embedding. Hasil ini TIDAK otomatis')
print('berpindah ke deskriptor embedding.')

CAVEATS = [
    'phi(y) = geometri ruang OUTPUT dari matriks skor, BUKAN phi embedding Pl@ntNet.',
    'Penyimpangan dari urutan pre-registered: dataset ini dijalankan meski Phase 1 Pl@ntNet',
    '  tidak lulus. Alasan: kegagalannya terdiagnosis sebagai daya uji dan tidak bisa diangkat',
    '  di Pl@ntNet (38 kelas/stratum). Dicatat di prereg_imagenet_gate.md.',
    'ImageNet berimbang -> ablasi prevalensi HAMPA, dikeluarkan dari keluarga uji primer.',
    '  Gate C di sini menguji HANYA lawan baseline jarak (prof_knn_1, dicadangkan di luar',
    '  model penuh): lengan prior-art yang lebih penting, tetapi uji yang LEBIH SEMPIT.',
    'RUN GERBANG, bukan run paper. Dataset ini jadi dataset paper hanya jika lulus, dan angka',
    '  paper harus dari porsi evaluasi yang tidak tersentuh keputusan gerbang.',
]
for c in CAVEATS: print('CAVEAT:', c)

path = write_report(
    name='05_imagenet_gate',
    hypothesis='delta_y is predictable from CLASS-LEVEL OUTPUT-SPACE geometry beyond trivial '
               'predictors, at a class count adequate to detect a weak effect',
    pass_criteria='PRIMARY per reports/prereg_imagenet_gate.md: stable feature set, n_cal=25, '
                  'alpha=0.10, all classes unstratified. Gate B = class-level bootstrap CI low '
                  '> 0 (n_boot=400). Gate C = class-level permutation null (n_perm=1000) vs '
                  'log_prevalence_only and distance_only_prereg, Holm-Bonferroni corrected. '
                  'Split-level CIs and the CI-excludes-0 criterion are NOT used: the unit of '
                  'exchangeability is the class, and 0 is not the null of the paired difference.',
    config=dict(dataset=DATASET, n_classes=K, alpha_primary=ALPHA_PRIMARY,
                n_cal_primary=N_CAL_PRIMARY, n_boot_class=N_BOOT_CLASS,
                n_perm_class=N_PERM_CLASS, alphas_secondary=list(ALPHAS_SECONDARY),
                n_cal_secondary=N_CAL_SECONDARY, stable_threshold=STABLE_THRESHOLD,
                frac_desc=FRAC_DESC, frac_cal=FRAC_CAL,
                descriptor_family='output_space', feature_sets=clean(FEATURE_SETS),
                scores_source='CCC released dump (no forward pass)',
                amendments=['#amendment-10']),
    seed=SEED,
    results=clean(dict(premise_ok=PREMISE_OK, dump_accuracy=acc,
                       samples_per_class=dict(min=int(cnt.min()), median=float(np.median(cnt)),
                                              max=int(cnt.max())),
                       split_sizes=dict(desc=len(i_desc), cal=len(i_cal), eval=len(i_eval)),
                       stability=stab, stable_set=stable_names,
                       gate_A=dict(reliability=r_delta, ci=[ciA['ci_low'], ciA['ci_high']],
                                   pass_=gate_A_pass),
                       ceilings=dict(r_delta=r_delta, r_phi=r_phi, joint=r_delta*r_phi),
                       gate_B_class_bootstrap=boot, gate_B_split_secondary=sf,
                       gate_C_permutation=perm_res, gate_C_holm=holm,
                       gate_C_ablations_tested=ABLATIONS_PRIMARY,
                       prevalence_ablation_degenerate=bool(spl['prevalence_ablation_degenerate']),
                       distance_baseline=DISTANCE_BASELINE,
                       gate_B_pass=gate_B_pass, gate_C_pass=gate_C_pass,
                       multi_alpha=multi, secondary=sec, sec_6_4=sec64,
                       clustered_cp=clustered, caveats=CAVEATS)),
    conclusion=verdict,
    runtime_seconds=0.0)
print()
print('laporan:', path)
print('VERDICT:', verdict)
